
# BRONZE LAYER Raw Data Ingestion In Database
The Bronze layer is the first step in our pipeline where we are taking raw csv file and storing it in database.  specifically MongoDB. with out 
touching it no cleaning no transformation just storing it as it is.
*so if something goes worng later we do have a source of truth*
# What does his notebook does it 
1. connects to the database
2. reads the csv file
3. Insert the raw data into the mongodb collection called bronze trips
4. verfes the rows and column counts

## Step 1: Import libraries & connect to MongoDB

In [ ]:
import os
import json
import warnings
import pandas as pd                       
from pymongo import MongoClient           
import time
warnings.filterwarnings('ignore')

# Data directory and file path
DATA_DIR = '/workspace/workdisk3/rudhra/Desktop/junks/big_data_project/data'
CSV_FILE = os.path.join(DATA_DIR, 'yellow_tripdata_2016-01.csv')

# Start MongoDB container
# docker run -d -p 27017:27017 --name my-mongo mongo:latest
client = MongoClient('mongodb://localhost:27017/')
db = client['nyc_taxi']  

print(f"Connected to MongoDB version {client.server_info()['version']}")
print(f"CSV file: {CSV_FILE}")
print(f"Existing collections: {db.list_collection_names()}")

Connected to MongoDB version 8.2.7
CSV file: /workspace/workdisk3/rudhra/Desktop/junks/big_data_project/data/yellow_tripdata_2016-01.csv
Existing collections: []


## Step 2: Drop old data & read the ENTIRE CSV into MongoDB

we are droping if something is in that database and we are inserting 100K rows chunk to reduce memory consumption


In [ ]:
# Drop ALL existing collections for a clean start
for col_name in db.list_collection_names():
    db[col_name].drop()
    print(f"  Dropped: {col_name}")


# this creates the collection on first insert
bronze_col = db['bronze_trips']    

# Columns that should be numeric (not strings)
numeric_cols = [
    'VendorID', 'passenger_count', 'trip_distance',
    'pickup_longitude', 'pickup_latitude',
    'dropoff_longitude', 'dropoff_latitude',
    'RatecodeID', 'payment_type',
    'fare_amount', 'extra', 'mta_tax',
    'tip_amount', 'tolls_amount',
    'improvement_surcharge', 'total_amount'
]


# rows per chunk to keeps memory under ~200MB
CHUNK_SIZE = 100_000    
total_inserted = 0
start_time = time.time()

print(f"\nReading ENTIRE CSV in chunks of {CHUNK_SIZE:,} rows...")
print(f"Estimated total rows: ~10,906,858")
print("-" * 60)

# pd.read_csv with chunksize returns an iterator each iteration gives a DataFrame of CHUNK_SIZE rows
for i, chunk in enumerate(pd.read_csv(CSV_FILE, chunksize=CHUNK_SIZE, low_memory=False)):
    
    # Step A: Filter out any repeated header rows that sometimes appear in the middle of the CSV
    chunk = chunk[chunk['tpep_pickup_datetime'] != 'tpep_pickup_datetime']
    
    # Step B: Convert datetime columns → Python datetime objects
    # This is CRITICAL: MongoDB will store them as ISODate, which lets us use $hour, $dayOfWeek etc. in Silver layer
    chunk['tpep_pickup_datetime'] = pd.to_datetime(chunk['tpep_pickup_datetime'], format='mixed')
    chunk['tpep_dropoff_datetime'] = pd.to_datetime(chunk['tpep_dropoff_datetime'], format='mixed')
    
    # Step C: Convert numeric columns from strings to numbers
    # errors='coerce' turns bad values (like "abc") into NaN instead of crashing
    for col in numeric_cols:
        chunk[col] = pd.to_numeric(chunk[col], errors='coerce')
    
    # Step D: Convert DataFrame → list of dicts (one dict = one MongoDB document)
    records = chunk.to_dict(orient='records')
    
    # Step E: Insert this chunk into MongoDB
    bronze_col.insert_many(records)
    
    # Track progress
    total_inserted += len(chunk)
    elapsed = time.time() - start_time
    rate = total_inserted / elapsed if elapsed > 0 else 0
    
    # Print progress every 500K rows
    if total_inserted % 500_000 < CHUNK_SIZE:
        eta = (10_906_858 - total_inserted) / rate if rate > 0 else 0
        print(f"  {total_inserted:>10,} rows inserted  ({rate:,.0f} rows/sec)  ETA: {eta/60:.1f} min")

elapsed = time.time() - start_time
print("-" * 60)
print(f"\nDONE! Inserted {total_inserted:,} rows in {elapsed/60:.1f} minutes ({total_inserted/elapsed:,.0f} rows/sec)")


Reading ENTIRE CSV in chunks of 100,000 rows...
Estimated total rows: ~10,906,858
------------------------------------------------------------


     500,000 rows inserted  (55,201 rows/sec)  ETA: 3.1 min


   1,000,000 rows inserted  (55,933 rows/sec)  ETA: 3.0 min


   1,500,000 rows inserted  (56,071 rows/sec)  ETA: 2.8 min


   2,000,000 rows inserted  (56,115 rows/sec)  ETA: 2.6 min


...

   9,000,000 rows inserted  (55,498 rows/sec)  ETA: 0.6 min


   9,500,000 rows inserted  (55,466 rows/sec)  ETA: 0.4 min


  10,000,000 rows inserted  (55,426 rows/sec)  ETA: 0.3 min


  10,500,000 rows inserted  (55,353 rows/sec)  ETA: 0.1 min


------------------------------------------------------------

DONE! Inserted 10,906,858 rows in 3.3 minutes (55,342 rows/sec)
